# 03. 양자화 + 대회 채점 + 속도 측정

02에서 고른 모델로 **최적화 전 / 후**를 비교하는 표를 만들어요. 예선 보고서의 핵심 결과예요.

| 모델 | 설명 |
| --- | --- |
| FP32 | 최적화 전 원본 (베이스라인) |
| INT8 (전체) | 모든 계산을 INT8로 |
| INT8 (헤드 FP32) | 박스 계산 부분만 FP32로 남기고 나머지는 INT8 (혼합 정밀도) |

재는 것: **KITTI Moderate AP40** (대회 공식 점수), 유지율, 파라미터 수, FLOPs, 모델 크기, CPU latency, 메모리

- GPU 없이 CPU 런타임으로도 돌아가요. 대략 20~30분 걸려요.
- 시험지(eval) 1,000장은 **여기서 처음이자 마지막으로** 써요.

> **NMS 기본값은 0.5예요.** 처음엔 0.7로 실험했고, v8n 실험 때 dev로 비교해 0.5로 정했어요 (그 비교 칸은 지금은 빠져 있고, git 기록에 남아 있어요). 그래서 지금 5~9번을 다시 돌리면 예전 표(0.7 기준)와 숫자가 조금 달라요.

## 1. Google Drive 연결 + 데이터 준비 + 도구 설치

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/embeded_ondeviceAI/runs'

In [ ]:
%%bash
set -e   # 중간에 실패하면 바로 멈추고 어디서 실패했는지 보여 줌
git clone -q https://github.com/gyu-young-04/embeded_ondeviceAI.git /content/embeded_ondeviceAI 2>/dev/null || git -C /content/embeded_ondeviceAI pull -q
mkdir -p /content/kitti && cd /content/kitti
for f in data_object_label_2.zip data_object_image_2.zip; do
  wget -c -q https://s3.eu-central-1.amazonaws.com/avg-kitti/$f
  unzip -q -n $f 'training/*'
done
[ "$(ls training/image_2 | wc -l)" = 7481 ] || { echo '사진이 7481장이 아니에요. /content/kitti 를 지우고 다시 실행하세요.'; exit 1; }
cd /content/embeded_ondeviceAI
[ -f /content/kitti_yolo/kitti.yaml ] || python scripts/prepare_kitti.py --kitti /content/kitti --out /content/kitti_yolo

In [ ]:
!pip -q install ultralytics==8.4.189 onnxruntime==1.31.0 onnx onnxslim nvidia-modelopt==0.48.0   # 버전 고정 (재현용)
import sys
sys.path.insert(0, '/content/embeded_ondeviceAI/scripts')
import ultralytics, onnxruntime, torch
print('ultralytics', ultralytics.__version__, '| onnxruntime', onnxruntime.__version__, '| torch', torch.__version__)

## 2. 어떤 모델을 최적화할지 고르기

02 노트북 비교표에서 제일 좋았던 실험 이름을 넣어요.

In [ ]:
NAME = 'yolov8n_960_e20'   # 본선용: 'yolo11n_1152_e90_kd_cw', 95점 도전: 'yolo11n_1280_e100_kd_cw1.0_full', 사람 점수: 'yolo11n_1152_e90_kd_cw_ped'

from pathlib import Path
run = Path(RUNS) / NAME
best_pt = run / 'weights' / 'best.pt'
assert best_pt.exists(), f'{best_pt} 가 없어요. 02 노트북에서 학습이 끝났는지 확인하세요.'

IMGSZ = int(NAME.split('_')[1])
W = IMGSZ
H = round(IMGSZ * 375 / 1242 / 32) * 32   # KITTI 비율에 맞춘 32의 배수 (960 → 288)
OUT = run / 'optimize'
OUT.mkdir(exist_ok=True)
print(f'입력 크기: {W} x {H}, 결과 폴더: {OUT}')

## 3. ONNX로 변환 (FP32 베이스라인)

입력 크기를 고정해서(static shape) 내보내요. Hailo 컴파일러도 고정 크기를 써요.

In [ ]:
import shutil
from ultralytics import YOLO

fp32 = OUT / 'fp32.onnx'
if not fp32.exists():
    f = YOLO(str(best_pt)).export(format='onnx', imgsz=[H, W], opset=13, simplify=True, dynamic=False)
    shutil.copy(f, fp32)
print(fp32)

## 4. INT8 양자화 (PTQ)

- Calibration(눈금 맞추기)에는 **학습(train) 사진 300장**을 써요. 시험지는 쓰지 않아요.
- 두 가지를 만들어서 비교해요: 전체 INT8 / 박스 계산 부분만 FP32.

In [ ]:
%cd /content/embeded_ondeviceAI
!python scripts/quantize_onnx.py --model {OUT}/fp32.onnx --out {OUT}/int8_full.onnx --calib-list /content/kitti_yolo/train.txt
!python scripts/quantize_onnx.py --model {OUT}/fp32.onnx --out {OUT}/int8_head_fp32.onnx --calib-list /content/kitti_yolo/train.txt --keep-head-fp32

## 5. 대회 채점 (KITTI Moderate AP40, 시험지 1,000장)

모델마다 1,000장을 추론해서 KITTI 형식 예측 파일을 만들고 공식 방식으로 채점해요. 모델당 몇 분 걸려요.

In [ ]:
import onnx_infer, kitti_eval

MODELS = {'FP32': 'fp32.onnx', 'INT8 (전체)': 'int8_full.onnx', 'INT8 (헤드 FP32)': 'int8_head_fp32.onnx'}
ids = [l.strip() for l in open('/content/embeded_ondeviceAI/eval_val.txt') if l.strip()]
scores = {}
for label, fname in MODELS.items():
    pred_dir = f'/content/preds/{NAME}/{fname[:-5]}'
    onnx_infer.predict_folder(OUT / fname, '/content/kitti/training/image_2', ids, pred_dir)
    result, mod_map = kitti_eval.evaluate(pred_dir, '/content/kitti/training/label_2', ids)
    scores[label] = (result, mod_map)
    print(f'[{label}]')
    print(kitti_eval.format_result(result, mod_map), '\n')

## 6. 속도 · 메모리 측정 (CPU, batch 1)

- 같은 컴퓨터에서 warm-up 10번 후 100번 재서 평균을 내요. 모델 실행 시간만 재요 (전처리·NMS 제외).
- 모델마다 새로 프로그램을 띄워서 재야 메모리 값이 섞이지 않아요.

In [ ]:
import json, os, subprocess

threads = os.cpu_count()
bench = {}
for label, fname in MODELS.items():
    out = subprocess.run(['python', 'scripts/benchmark_onnx.py', '--model', str(OUT / fname), '--threads', str(threads)],
                         capture_output=True, text=True, check=True).stdout
    bench[label] = json.loads(out.strip().splitlines()[-1])
    print(label, bench[label])
!lscpu | grep 'Model name'

## 7. 최종 비교표

이 표가 보고서의 **Baseline 대비 정량 비교 결과**예요. Drive에 `results.csv` 로도 저장돼요.

In [ ]:
import pandas as pd
from ultralytics.utils.torch_utils import get_flops

torch_model = YOLO(str(best_pt)).model
params = sum(p.numel() for p in torch_model.parameters())
gflops = get_flops(torch_model, [H, W])
base = scores['FP32'][1]

rows = []
for label, fname in MODELS.items():
    result, mod_map = scores[label]
    b = bench[label]
    rows.append({
        '모델': label,
        'Car': round(result['Car'][1], 2),
        'Pedestrian': round(result['Pedestrian'][1], 2),
        'Cyclist': round(result['Cyclist'][1], 2),
        'Moderate mAP': round(mod_map, 2),
        '유지율(%)': round(mod_map / base * 100, 1),
        '파라미터(M)': round(params / 1e6, 2),
        'GFLOPs': round(gflops, 2),
        '파일 크기(MB)': round((OUT / fname).stat().st_size / 1e6, 1),
        'latency(ms)': f"{b['latency_ms_mean']} ± {b['latency_ms_std']}",
        'latency 중앙값(ms)': b['latency_ms_p50'],
        'FPS': b['fps'],
        '메모리(MB)': b['model_mem_mb'],
    })

table = pd.DataFrame(rows)
table.to_csv(OUT / 'results.csv', index=False)
print(f'{NAME} | 입력 {W}x{H} | CPU {threads}스레드 | KITTI Moderate AP40 (eval_val 1,000장)')
table

### 표 읽는 법

- **유지율**: 원본 점수의 몇 %를 지켰는지. 100%에 가까울수록 좋아요.
- **파라미터 / GFLOPs**: 양자화는 숫자의 길이만 줄이고 개수는 그대로라서 세 모델 모두 같아요.
- **파일 크기, latency**: 작을수록 좋아요. INT8은 보통 크기 약 1/4, 속도 약 2배.
- **INT8 (전체)** 점수가 크게 떨어지고 **INT8 (헤드 FP32)** 는 괜찮다면, "박스 좌표와 점수를 한 숫자 범위로 줄이면 점수가 뭉개져서, 그 부분만 FP32로 남겼다"가 보고서의 핵심 분석 포인트가 돼요.

## 8. 눈금(Calibration) 방식 비교 (선택, CPU로 약 1시간)

INT8로 바꿀 때 '숫자 범위를 어떻게 정할지'를 3가지로 비교해요. 모두 헤드 FP32 기준이에요.

| 방식 | 범위 정하는 법 |
| --- | --- |
| MinMax | 가장 작은 값 ~ 가장 큰 값 (지금까지 쓴 방식) |
| Percentile | 너무 튀는 값(상위 0.001%)은 버리고 범위를 좁게 |
| Entropy | 원래 값 분포와 가장 비슷해지는 범위를 계산으로 찾기 |

- 비교는 **쪽지시험(dev 648장)** 으로 해요. 시험지(eval)로 고르면 커닝이에요.
- 결과는 `calib_compare_dev.csv` 로 저장돼요.

In [ ]:
import json, os, subprocess
import pandas as pd
os.chdir('/content/embeded_ondeviceAI')

METHODS = {'MinMax': 'int8_head_fp32.onnx',
           'Percentile': 'int8_head_fp32_percentile.onnx',
           'Entropy': 'int8_head_fp32_entropy.onnx'}
for m, fname in METHODS.items():
    if not (OUT / fname).exists():
        print(m, '양자화 중...')
        subprocess.run(['python', 'scripts/quantize_onnx.py', '--model', str(OUT / 'fp32.onnx'), '--out', str(OUT / fname),
                        '--calib-list', '/content/kitti_yolo/train.txt', '--keep-head-fp32', '--method', m], check=True)

# 쪽지시험(dev 648장)으로 채점 — 시험지(eval)로 고르면 커닝
dev_ids = [Path(l.strip()).stem for l in open('/content/kitti_yolo/dev.txt') if l.strip()]
def dev_score(fname):
    pred_dir = f'/content/preds_dev/{NAME}/{fname[:-5]}'
    onnx_infer.predict_folder(OUT / fname, '/content/kitti/training/image_2', dev_ids, pred_dir)
    return kitti_eval.evaluate(pred_dir, '/content/kitti/training/label_2', dev_ids)

dev_base = dev_score('fp32.onnx')[1]
rows = []
for m, fname in METHODS.items():
    result, mod_map = dev_score(fname)
    rows.append({'눈금 방식': m,
                 'Car': round(result['Car'][1], 2),
                 'Pedestrian': round(result['Pedestrian'][1], 2),
                 'Cyclist': round(result['Cyclist'][1], 2),
                 'dev Moderate mAP': round(mod_map, 2),
                 'dev 유지율(%)': round(mod_map / dev_base * 100, 1)})
calib = pd.DataFrame(rows)
calib.to_csv(OUT / 'calib_compare_dev.csv', index=False)
print(f'FP32 dev Moderate mAP: {dev_base:.2f}')
calib

## 9. 고른 방식으로 최종 채점

dev에서 제일 좋았던 방식 **하나만** 시험지(eval 1,000장)로 채점해서 7번 표에 한 줄 추가해요.

> 이름이 `_full` 로 끝나는 모델(dev 도 학습에 넣음)은 dev 로 고르면 공정하지 않아서, 예전 모델들에서 이긴 **Percentile** 을 그대로 써요.

In [ ]:
if '_full' in NAME:
    BEST = 'Percentile'   # dev 를 학습에 썼으니 dev 로 고르지 않아요
    print('dev 를 학습에 쓴 모델이라 예전에 고른 방식을 그대로 써요:', BEST)
else:
    BEST = calib.loc[calib['dev Moderate mAP'].idxmax(), '눈금 방식']
    print('dev 점수로 고른 방식:', BEST)

if BEST == 'MinMax':
    print('MinMax 가 제일 좋아요. 7번 표가 그대로 최종이에요.')
else:
    fname = METHODS[BEST]
    pred_dir = f'/content/preds/{NAME}/{fname[:-5]}'
    onnx_infer.predict_folder(OUT / fname, '/content/kitti/training/image_2', ids, pred_dir)
    result, mod_map = kitti_eval.evaluate(pred_dir, '/content/kitti/training/label_2', ids)
    print(kitti_eval.format_result(result, mod_map))
    out = subprocess.run(['python', 'scripts/benchmark_onnx.py', '--model', str(OUT / fname), '--threads', str(threads)],
                         capture_output=True, text=True, check=True).stdout
    b = json.loads(out.strip().splitlines()[-1])
    row = {
        '모델': f'INT8 (헤드 FP32, {BEST})',
        'Car': round(result['Car'][1], 2),
        'Pedestrian': round(result['Pedestrian'][1], 2),
        'Cyclist': round(result['Cyclist'][1], 2),
        'Moderate mAP': round(mod_map, 2),
        '유지율(%)': round(mod_map / base * 100, 1),
        '파라미터(M)': round(params / 1e6, 2),
        'GFLOPs': round(gflops, 2),
        '파일 크기(MB)': round((OUT / fname).stat().st_size / 1e6, 1),
        'latency(ms)': f"{b['latency_ms_mean']} ± {b['latency_ms_std']}",
        'latency 중앙값(ms)': b['latency_ms_p50'],
        'FPS': b['fps'],
        '메모리(MB)': b['model_mem_mb'],
    }
    table = pd.concat([table[~table['모델'].str.startswith(f'INT8 (헤드 FP32, ')], pd.DataFrame([row])], ignore_index=True)
    table.to_csv(OUT / 'results.csv', index=False)
table

## 13. 이어서 하기 (세션이 끊겼을 때만)

14·15번은 5~7번에서 만든 값(점수표, FP32 점수 등)을 써요. 세션이 끊긴 뒤에는 5~7번을 다시 돌리지 않고
**Drive에 저장된 결과표(results.csv)로 복구**할 수 있어요.

순서: **1번(3칸) → 2번(NAME) → 이 칸 → 14번부터**

- 세션이 안 끊겼으면 이 칸은 건너뛰어도 돼요.
- 속도(latency)는 새 세션의 CPU로 재요. 표의 예전 줄과는 컴퓨터가 달라서 **속도 비교는 같은 세션 안에서만** 하세요.


In [ ]:
import json, os, subprocess, sys
import pandas as pd
sys.path.insert(0, '/content/embeded_ondeviceAI/scripts')
import onnx_infer, kitti_eval
os.chdir('/content/embeded_ondeviceAI')

assert (OUT / 'results.csv').exists(), f'{OUT}/results.csv 가 없어요. 3~7번을 먼저 실행하세요.'
table = pd.read_csv(OUT / 'results.csv')
ids = [l.strip() for l in open('/content/embeded_ondeviceAI/eval_val.txt') if l.strip()]
threads = os.cpu_count()
fp32_row = table[table['모델'] == 'FP32'].iloc[0]
base = float(fp32_row['Moderate mAP'])
params = float(fp32_row['파라미터(M)']) * 1e6
gflops = float(fp32_row['GFLOPs'])

# 14-2번을 이미 했으면 그때 고른 블록 수도 복구 (15번 비교표에 들어가요)
if (OUT / 'sensitivity_topk_dev.csv').exists():
    topk = pd.read_csv(OUT / 'sensitivity_topk_dev.csv')
    BEST_K = int(topk.loc[topk['dev Moderate mAP'].idxmax(), 'FP32 블록 수'])
    print('14-2번 결과 복구: FP32 블록 수 =', BEST_K)
print(f'복구 완료: {NAME} | FP32 {base} | 저장된 줄 {len(table)}개')
table


## 14. 층별 민감도 분석 (본선용, CPU로 약 30~40분)

블록(층 묶음)을 **하나씩만 FP32로 남기고** 나머지는 INT8로 바꿔서 점수를 재요.
점수가 많이 회복되는 블록 = INT8에 약한 블록. 이런 블록 몇 개만 FP32로 두면 크기·속도는 거의 그대로, 점수는 회복돼요.

- 눈금(Percentile)은 처음 한 번만 재서 저장해 두고 재사용해요.
- 순위는 dev 300장으로 빠르게 매기고, 실제 선택은 다음 칸에서 dev 전체로 해요.
- **먼저 1~9번을 실행**해 두세요 (fp32.onnx, 7번 표가 필요해요).
- 이름이 `_full` 로 끝나는 모델은 이 칸을 **건너뛰어요.** 14-2번이 예전 모델(11n 1152)에서 찾은 블록 [0, 2]를 그대로 써요.


In [ ]:
import json, os, subprocess
import pandas as pd
os.chdir('/content/embeded_ondeviceAI')
METHOD = 'Percentile'          # 8번 칸에서 이긴 눈금 방식
SENS = OUT / 'sensitivity'

if not (SENS / 'sensitivity.csv').exists():
    cmd = (f'python scripts/sensitivity.py --model {OUT}/fp32.onnx --calib-list /content/kitti_yolo/train.txt '
           f'--dev-list /content/kitti_yolo/dev.txt --images /content/kitti/training/image_2 '
           f'--labels /content/kitti/training/label_2 --out-dir {SENS} --method {METHOD}')
    !{cmd}
print(open(SENS / 'sensitivity_base.txt').read())
sens = pd.read_csv(SENS / 'sensitivity.csv')
sens.head(10)   # 위에 있을수록 INT8에 약한 블록


## 14-2. 약한 블록 몇 개를 FP32로 둘지 고르기

약한 순서대로 0·1·2·3·5개를 FP32로 두고 **dev 전체(648장)** 로 비교해요. 제일 좋은 것 하나만 시험지로 채점해서 7번 표에 추가해요.
FP32로 두는 블록이 많을수록 점수는 오르지만 파일이 커지고 느려져요 → 크기 열도 같이 보세요.

> `_full` 모델은 dev 로 고르면 공정하지 않아서, 예전 모델(11n 1152)에서 고른 **블록 [0, 2]** 를 그대로 FP32로 둬요.


In [ ]:
# 시험지(eval) 채점 + 속도 측정 후 7번 표(results.csv)에 한 줄 추가하는 공통 함수
def add_result(label, fname):
    global table
    pred_dir = f'/content/preds/{NAME}/{fname[:-5]}'
    onnx_infer.predict_folder(OUT / fname, '/content/kitti/training/image_2', ids, pred_dir)
    result, mod_map = kitti_eval.evaluate(pred_dir, '/content/kitti/training/label_2', ids)
    print(f'[{label}]')
    print(kitti_eval.format_result(result, mod_map))
    out = subprocess.run(['python', 'scripts/benchmark_onnx.py', '--model', str(OUT / fname), '--threads', str(threads)],
                         capture_output=True, text=True, check=True).stdout
    b = json.loads(out.strip().splitlines()[-1])
    row = {'모델': label, 'Car': round(result['Car'][1], 2), 'Pedestrian': round(result['Pedestrian'][1], 2),
           'Cyclist': round(result['Cyclist'][1], 2), 'Moderate mAP': round(mod_map, 2),
           '유지율(%)': round(mod_map / base * 100, 1), '파라미터(M)': round(params / 1e6, 2), 'GFLOPs': round(gflops, 2),
           '파일 크기(MB)': round((OUT / fname).stat().st_size / 1e6, 1),
           'latency(ms)': f"{b['latency_ms_mean']} ± {b['latency_ms_std']}", 'latency 중앙값(ms)': b['latency_ms_p50'],
           'FPS': b['fps'], '메모리(MB)': b['model_mem_mb']}
    table = pd.concat([table[table['모델'] != label], pd.DataFrame([row])], ignore_index=True)
    table.to_csv(OUT / 'results.csv', index=False)
    return mod_map


# 쪽지시험(dev 648장) 채점 — 고를 때는 dev 만 써요
dev_ids = [Path(l.strip()).stem for l in open('/content/kitti_yolo/dev.txt') if l.strip()]
def dev_score(fname):
    pred_dir = f'/content/preds_dev/{NAME}/{fname[:-5]}'
    onnx_infer.predict_folder(OUT / fname, '/content/kitti/training/image_2', dev_ids, pred_dir)
    return kitti_eval.evaluate(pred_dir, '/content/kitti/training/label_2', dev_ids)

METHOD = globals().get('METHOD', 'Percentile')
SENS = OUT / 'sensitivity'
SENS.mkdir(exist_ok=True)
if '_full' in NAME:
    order, KS = [0, 2], [2]   # dev 를 학습에 썼으니 예전 모델에서 고른 블록을 그대로
else:
    order, KS = sens['블록'].tolist(), [0, 1, 2, 3, 5]
rows = []
for k in KS:
    blocks = order[:k]
    f = f'int8_sens_top{k}.onnx'
    if not (OUT / f).exists():
        subprocess.run(['python', 'scripts/quantize_onnx.py', '--model', str(OUT / 'fp32.onnx'), '--out', str(OUT / f),
                        '--calib-list', '/content/kitti_yolo/train.txt', '--keep-head-fp32', '--method', METHOD,
                        '--calib-cache', str(SENS / f'calib_{METHOD.lower()}.json'),
                        '--fp32-blocks', *map(str, blocks)], check=True)
    result, m = dev_score(f)
    rows.append({'FP32 블록 수': k, '블록': blocks, 'dev Moderate mAP': round(m, 2),
                 'Pedestrian': round(result['Pedestrian'][1], 2), 'Cyclist': round(result['Cyclist'][1], 2),
                 '파일 크기(MB)': round((OUT / f).stat().st_size / 1e6, 1)})
topk = pd.DataFrame(rows)
topk.to_csv(OUT / 'sensitivity_topk_dev.csv', index=False)
display(topk)

BEST_K = int(topk.loc[topk['dev Moderate mAP'].idxmax(), 'FP32 블록 수'])
print('dev 점수로 고른 FP32 블록 수:', BEST_K, order[:BEST_K])
if BEST_K > 0:
    add_result(f'INT8 (민감도, FP32 블록 {order[:BEST_K]})', f'int8_sens_top{BEST_K}.onnx')
table


## 15. QAT 모델 채점 (본선용)

02 노트북 8번 칸(QAT)이 끝난 뒤에 실행해요. `NAME` 은 QAT 출발점 실험(예: `yolo11n_1152_e90_kd_cw`)이어야 해요.
이 모델의 7번 표가 Drive 에 있으면 **1, 2번만 실행하고 바로 이 칸**을 돌려도 돼요.

1. QAT 모델을 ONNX(INT8)로 내보내요. 눈금은 학습 중에 이미 정해져 있어서 따로 재지 않아요.
2. 내보낸 파일은 가중치가 FP32로 저장돼 있어서, **가중치만 진짜 INT8로 바꿔 저장**해요 (결과는 똑같고 크기만 약 1/4).
3. dev로 PTQ와 비교하고, 시험지로 채점해서 7번 표에 추가해요.


In [ ]:
# 1, 2번만 실행하고 바로 왔을 때: 7번에서 저장한 표로 준비해요 (3~14번 생략 가능)
import json, os, subprocess
import pandas as pd
import onnx_infer, kitti_eval
os.chdir('/content/embeded_ondeviceAI')
if 'table' not in globals():
    assert (OUT / 'results.csv').exists(), '7번 표가 없어요. 이 모델은 1~7번을 먼저 실행하세요.'
    ids = [l.strip() for l in open('eval_val.txt') if l.strip()]
    threads = os.cpu_count()
    table = pd.read_csv(OUT / 'results.csv')
    r = table[table['모델'] == 'FP32'].iloc[0]
    base, params, gflops = r['Moderate mAP'], r['파라미터(M)'] * 1e6, r['GFLOPs']

# 시험지(eval) 채점 + 속도 측정 후 7번 표(results.csv)에 한 줄 추가하는 공통 함수
def add_result(label, fname):
    global table
    pred_dir = f'/content/preds/{NAME}/{fname[:-5]}'
    onnx_infer.predict_folder(OUT / fname, '/content/kitti/training/image_2', ids, pred_dir)
    result, mod_map = kitti_eval.evaluate(pred_dir, '/content/kitti/training/label_2', ids)
    print(f'[{label}]')
    print(kitti_eval.format_result(result, mod_map))
    out = subprocess.run(['python', 'scripts/benchmark_onnx.py', '--model', str(OUT / fname), '--threads', str(threads)],
                         capture_output=True, text=True, check=True).stdout
    b = json.loads(out.strip().splitlines()[-1])
    row = {'모델': label, 'Car': round(result['Car'][1], 2), 'Pedestrian': round(result['Pedestrian'][1], 2),
           'Cyclist': round(result['Cyclist'][1], 2), 'Moderate mAP': round(mod_map, 2),
           '유지율(%)': round(mod_map / base * 100, 1), '파라미터(M)': round(params / 1e6, 2), 'GFLOPs': round(gflops, 2),
           '파일 크기(MB)': round((OUT / fname).stat().st_size / 1e6, 1),
           'latency(ms)': f"{b['latency_ms_mean']} ± {b['latency_ms_std']}", 'latency 중앙값(ms)': b['latency_ms_p50'],
           'FPS': b['fps'], '메모리(MB)': b['model_mem_mb']}
    table = pd.concat([table[table['모델'] != label], pd.DataFrame([row])], ignore_index=True)
    table.to_csv(OUT / 'results.csv', index=False)
    return mod_map


# 쪽지시험(dev 648장) 채점 — 고를 때는 dev 만 써요
dev_ids = [Path(l.strip()).stem for l in open('/content/kitti_yolo/dev.txt') if l.strip()]
def dev_score(fname):
    pred_dir = f'/content/preds_dev/{NAME}/{fname[:-5]}'
    onnx_infer.predict_folder(OUT / fname, '/content/kitti/training/image_2', dev_ids, pred_dir)
    return kitti_eval.evaluate(pred_dir, '/content/kitti/training/label_2', dev_ids)

from ultralytics import YOLO
QAT_EPOCHS = 10
QAT_TAG = ''   # QAT 를 다른 설정으로 다시 했으면 그 꼬리표 (예: '_qs0.5')
qat_pt = Path(RUNS) / f'{NAME}_qat{QAT_EPOCHS}{QAT_TAG}' / 'weights' / 'best.pt'
assert qat_pt.exists(), f'{qat_pt} 가 없어요. 02 노트북 8번 칸(QAT)을 먼저 실행하세요.'

QAT_FILE = f'int8_qat{QAT_TAG}.onnx'
if not (OUT / QAT_FILE).exists():
    f = YOLO(str(qat_pt)).export(format='onnx', imgsz=[H, W], opset=13, quantize=8, dynamic=False)
    subprocess.run(['python', 'scripts/fold_qat_weights.py', '--model', f, '--out', str(OUT / QAT_FILE)], check=True)

cmp = {'FP32': 'fp32.onnx', 'PTQ (Percentile)': 'int8_head_fp32_percentile.onnx', 'QAT': QAT_FILE}
if globals().get('BEST_K'):
    cmp[f'PTQ + 민감도 (블록 {BEST_K}개)'] = f'int8_sens_top{BEST_K}.onnx'
cmp = {k: v for k, v in cmp.items() if (OUT / v).exists()}   # 안 만든 파일은 비교에서 빼요
rows = []
for label, f in cmp.items():
    result, m = dev_score(f)
    rows.append({'방식': label, 'dev Moderate mAP': round(m, 2), 'Pedestrian': round(result['Pedestrian'][1], 2),
                 'Cyclist': round(result['Cyclist'][1], 2), '파일 크기(MB)': round((OUT / f).stat().st_size / 1e6, 1)})
qat_cmp = pd.DataFrame(rows)
qat_cmp.to_csv(OUT / 'qat_compare_dev.csv', index=False)
display(qat_cmp)

add_result(f'INT8 (QAT{QAT_TAG})', QAT_FILE)
table


## 16. 클래스별 NMS 조정 (선택, 재학습 없음, CPU)

겹친 박스를 지우는 기준(NMS IoU)을 지금은 모든 클래스가 0.5 로 같이 써요. 이걸 클래스마다 따로 골라 봐요.

- 값은 **쪽지시험(dev 648장)** 으로만 골라요. 시험지(eval)는 고른 값을 마지막에 한 번 채점할 때만 써요.
- dev 에서 0.05 도 안 오르면 우연일 수 있어서 0.5 를 그대로 둬요.
- 15번까지 끝난 뒤에 실행해요. 결과는 `nms_tuned.json` 으로 저장돼요.
- 이름에 `_full` 이 있는 모델은 dev 를 학습에 썼으니 여기서 고르지 않아요.

In [ ]:
import os, subprocess
os.chdir('/content/embeded_ondeviceAI')
MODEL_FILE = 'int8_qat.onnx'   # 최종 후보 모델
assert '_full' not in NAME, 'dev 를 학습에 쓴 모델은 dev 로 고르면 안 돼요. _full 이 없는 같은 설정 모델에서 고른 값을 쓰세요.'
assert (OUT / MODEL_FILE).exists(), f'{MODEL_FILE} 가 없어요. 15번을 먼저 실행하세요.'
subprocess.run(['python', 'scripts/tune_nms.py', '--model', str(OUT / MODEL_FILE),
                '--images', '/content/kitti/training/image_2', '--labels', '/content/kitti/training/label_2',
                '--dev-list', '/content/kitti_yolo/dev.txt', '--work', f'/content/nms/{NAME}',
                '--out', str(OUT / 'nms_tuned.json')], check=True)